<a href="https://colab.research.google.com/github/Sangeetha3315/Agentic-AI-and-computer-vision-workshop-projects/blob/main/Hand_Gesture_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow opencv-python-headless matplotlib pillow

import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import io
from PIL import Image
import os, shutil, time

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

print("Webcam capture function ready. Test it:")
test_path = take_photo('test.jpg')
img = plt.imread(test_path)
plt.imshow(img)
plt.axis('off')
plt.title("Webcam test capture")
plt.show()

In [ ]:
img_bgr = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print("Image shape (Height, Width, Channels):", img_rgb.shape)
print("Pixel value at center of image (R,G,B):", img_rgb[img_rgb.shape[0]//2, img_rgb.shape[1]//2])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0,0].imshow(img_rgb); axes[0,0].set_title("Original (RGB)"); axes[0,0].axis('off')
axes[0,1].imshow(img_rgb[:,:,0], cmap='Reds'); axes[0,1].set_title("Red channel"); axes[0,1].axis('off')
axes[0,2].imshow(img_rgb[:,:,1], cmap='Greens'); axes[0,2].set_title("Green channel"); axes[0,2].axis('off')
axes[1,0].imshow(img_rgb[:,:,2], cmap='Blues'); axes[1,0].set_title("Blue channel"); axes[1,0].axis('off')
axes[1,1].imshow(img_gray, cmap='gray'); axes[1,1].set_title("Grayscale"); axes[1,1].axis('off')
axes[1,2].imshow(img_hsv[:,:,0], cmap='hsv'); axes[1,2].set_title("Hue channel (HSV)"); axes[1,2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(edges, cmap='gray'); axes[1].set_title("Canny Edge Detection\n(zero training needed)"); axes[1].axis('off')
plt.tight_layout()
plt.show()

print("Notice: edges find BOUNDARIES, but have no idea what a 'hand' or 'thumb' is.")
print("That's the gap deep learning closes — next steps.")

In [ ]:
DATA_DIR = 'gesture_data'
CLASSES = ['thumbs_up', 'thumbs_down', 'neutral']
for c in CLASSES:
    os.makedirs(os.path.join(DATA_DIR, c), exist_ok=True)

# --------------------------------------
CLASS_NAME = 'neutral'
NUM_IMAGES = 15
# --------------------------------------

save_dir = os.path.join(DATA_DIR, CLASS_NAME)
existing = len(os.listdir(save_dir))

print(f"Collecting images for class: '{CLASS_NAME}'")
print(f"Click 'Capture' {NUM_IMAGES} times. Vary your hand angle/position slightly each time.")

for i in range(NUM_IMAGES):
    fname = os.path.join(save_dir, f'{existing + i}.jpg')
    take_photo(fname)
    print(f"Captured {i+1}/{NUM_IMAGES} -> {fname}")

print(f"\nDone. Total images for '{CLASS_NAME}':", len(os.listdir(save_dir)))

In [ ]:
for c in CLASSES:
    n = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"{c}: {n} images")

In [ ]:
IMG_SIZE = 224

def load_dataset(data_dir, classes):
    X, y = [], []
    for idx, c in enumerate(classes):
        folder = os.path.join(data_dir, c)
        for fname in os.listdir(folder):
            path = os.path.join(folder, fname)
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            y.append(idx)
    return np.array(X), np.array(y)

X, y = load_dataset(DATA_DIR, CLASSES)
print("Dataset shape:", X.shape, "Labels shape:", y.shape)

X_preprocessed = preprocess_input(X.astype(np.float32))
y_onehot = tf.keras.utils.to_categorical(y, num_classes=len(CLASSES))

# quick visual check of a few samples
fig, axes = plt.subplots(1, 6, figsize=(15,3))
for i, ax in enumerate(axes):
    idx = np.random.randint(0, len(X))
    ax.imshow(X[idx])
    ax.set_title(CLASSES[y[idx]])
    ax.axis('off')
plt.show()

In [ ]:
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          include_top=False,
                          weights='imagenet')
base_model.trainable = False  # freeze the pretrained backbone

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_preprocessed, y_onehot, epochs=12, batch_size=4, validation_split=0.2)

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend(); plt.title('Training Progress'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.show()

In [ ]:
def predict_gesture(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_input = preprocess_input(img_resized.astype(np.float32))
    img_input = np.expand_dims(img_input, axis=0)

    preds = model.predict(img_input, verbose=0)[0]
    predicted_class = CLASSES[np.argmax(preds)]
    confidence = np.max(preds)

    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Prediction: {predicted_class}  ({confidence*100:.1f}% confidence)")
    plt.show()

    print("Full probability breakdown:")
    for c, p in zip(CLASSES, preds):
        print(f"  {c}: {p*100:.1f}%")

    return predicted_class, confidence

live_path = take_photo('live_test.jpg')
predict_gesture(live_path)